## Feature Engineering

I imported the required libraries for feature engineering and preprocessing. Pandas and NumPy are used for data manipulation and numerical operations, while Scikit-learn provides functions for splitting the data, encoding categorical features, scaling numerical features, and applying these transformations using ColumnTransformer.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

print("Libraries imported successfully.")


In [ ]:
file_path = "ev_vs_petrol_dataset_v3.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")


## Listing Dataset Columns

Before performing feature engineering, I displayed all the column names with their serial numbers. This helped me understand the available variables and select the appropriate columns for creating new features

In [ ]:
print("Columns in dataset:")

for i, col in enumerate(df.columns, 1):
    print(i, "->", col)

## Create Total Car Sales

I created a new feature called calculated_total_sales by adding EV, petrol, and diesel sales. This gives an overall measure of vehicle sales for each record. However, since the calculation includes my target variable, EV sales, I will exclude this feature from the machine-learning input variables to prevent target leakage.


In [ ]:
df["calculated_total_sales"] = (
    df["ev_sales"]
    + df["petrol_car_sales"]
    + df["diesel_car_sales"]
)

print("New feature created: calculated_total_sales")

df[
    [
        "ev_sales",
        "petrol_car_sales",
        "diesel_car_sales",
        "calculated_total_sales"
    ]
].head()

## Calculate EV Adoption Ratio
I created an EV adoption ratio by dividing EV sales by total vehicle sales. This provides a normalized measure of how strongly EVs are adopted in each market. However, because the calculation uses EV sales, which is my target variable, I will exclude this feature from the machine-learning predictors to prevent target leakage.

In [ ]:
df["ev_adoption_ratio"] = (
    df["ev_sales"]
    / df["calculated_total_sales"]
)

print("EV adoption ratio created.")

df[
    [
        "ev_sales",
        "calculated_total_sales",
        "ev_adoption_ratio"
    ]
].head()

## EV adoption percentage
I converted the EV adoption ratio into a percentage to make the result easier to interpret. This feature shows the percentage of total vehicle sales represented by EVs. Since it is calculated using EV sales, I will exclude it from the prediction features when building the EV sales machine-learning model.

In [ ]:
df["ev_adoption_percentage"] = (
    df["ev_adoption_ratio"] * 100
)

df[
    [
        "ev_sales",
        "ev_adoption_percentage"
    ]
].head()


## Petrol sales ratio
I created the petrol sales ratio by dividing petrol car sales by calculated total vehicle sales. This helps measure the relative share of petrol vehicles in each market. However, because the total sales calculation includes EV sales, I will exclude this feature from the EV sales prediction model to avoid target leakage.

In [ ]:
df["petrol_sales_ratio"] = (
    df["petrol_car_sales"] /
    df["calculated_total_sales"]
)

print(df["petrol_sales_ratio"].head())

## Diesel sales ratio
I created the diesel sales ratio by dividing diesel car sales by total vehicle sales. This helps measure the relative share of diesel vehicles in the market. However, because total vehicle sales includes EV sales, I will exclude this feature from the EV sales prediction model to avoid target leakage.


In [ ]:
df["diesel_sales_ratio"] = (
    df["diesel_car_sales"] /
    df["calculated_total_sales"]
)

print(df["diesel_sales_ratio"].head())

## Charging stations per EV sale
feature that measures charging stations relative to EV sales. I added 1 to EV sales to avoid division-by-zero errors. However, because this feature directly uses EV sales, which is my target variable, I will exclude it from the machine-learning predictors to prevent target leakage.

In [ ]:
df["charging_per_ev_sale"] = (
    df["charging_stations"] /
    (df["ev_sales"] + 1)
)

print(df["charging_per_ev_sale"].head())

## Create EV Price Category

I converted the average EV range into three categories: Low Range, Medium Range, and High Range. This makes the continuous range variable easier to interpret and allows me to analyse whether EV driving range is associated with EV adoption.

In [ ]:
df["ev_range_category"] = pd.cut(
    df["avg_ev_range_km"],
    bins=[0, 300, 500, np.inf],
    labels=[
        "Low Range",
        "Medium Range",
        "High Range"
    ]
)

print(df["ev_range_category"].value_counts())

## Fuel price category
I converted fuel price into three categories: Low, Medium, and High. I used qcut() so that the categories are created based on the distribution of the data, with approximately equal numbers of observations in each group. This makes the fuel-price feature easier to analyse.

In [ ]:
df["fuel_price_category"] = pd.qcut(
    df["fuel_price_usd_per_liter"],
    q=3,
    labels=[
        "Low",
        "Medium",
        "High"
    ],
    duplicates="drop"
)

print(df["fuel_price_category"].value_counts())

## Subsidy category
I converted EV subsidy values into three categories: Low, Medium, and High Subsidy using quantile-based binning. This makes subsidy levels easier to compare and helps analyse whether different subsidy levels are associated with EV sales. For machine learning, the category boundaries should be learned only from the training data to avoid data leakage


In [ ]:
df["subsidy_category"] = pd.qcut(
    df["ev_subsidy_usd"],
    q=3,
    labels=[
        "Low Subsidy",
        "Medium Subsidy",
        "High Subsidy"
    ],
    duplicates="drop"
)

print(df["subsidy_category"].value_counts())

## Infrastructure category
"I converted the number of charging stations into three categories: Low, Medium, and High Infrastructure using quantile-based classification. This makes the infrastructure variable easier to analyse and helps identify whether higher charging infrastructure is associated with stronger EV market performance.

In [ ]:
df["charging_category"] = pd.qcut(
    df["charging_stations"],
    q=3,
    labels=[
        "Low Infrastructure",
        "Medium Infrastructure",
        "High Infrastructure"
    ],
    duplicates="drop"
)

print(df["charging_category"].value_counts())

## Check newly created features
After completing feature engineering, I checked the shape of the updated dataset and displayed all column names. This allowed me to verify that the newly created features were successfully added and that the dataset structure was correct before proceeding to the next stage.

In [ ]:
print("New dataset shape:", df.shape)

print("\nAll columns:")

for col in df.columns:
    print(col)

## Check for infinity values
I checked the numerical columns for infinite values because some engineered features involve division. The check ensures that calculations such as ratios did not produce infinity due to zero denominators. A result of zero confirms that no infinite values are present.

In [ ]:
print(
    "Infinite values:",
    np.isinf(
        df.select_dtypes(include=np.number)
    ).sum().sum()
)

## Remove target leakage
"My target variable is ev_sales. Before training the model, I removed all features that were directly or indirectly derived from EV sales. This prevents target leakage and ensures that the model learns from independent variables rather than information that already contains the target.

In [ ]:
target = "ev_sales"

leakage_columns = [
    "ev_market_share",
    "ev_adoption_ratio",
    "ev_adoption_percentage",
    "is_ev_dominant",
    "calculated_total_sales",
    "petrol_sales_ratio",
    "diesel_sales_ratio",
    "charging_per_ev_sale"
]

X = df.drop(
    columns=[target] + leakage_columns,
    errors="ignore"
)

y = df[target]

print("Target:", target)
print("Features:", X.shape)
print("Target shape:", y.shape)

print("\nFeatures used for ML:")
for col in X.columns:
    print("-", col)

## Check final features

After removing the target and leakage variables, I displayed all the remaining features that will be used for machine learning. This step helps verify that only valid independent variables are included in the model.

In [ ]:

print("Features used for ML:")

for col in X.columns:
    print("-", col)

## Train/Test Split
After removing target leakage, I split the dataset into 80% training and 20% testing data. The training data is used to teach the model, while the testing data contains unseen observations used to evaluate the model's performance. I used random state 42 to make the split reproducible.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

## Identify numerical and categorical features
I separated the training features into numerical and categorical variables. Numerical variables will be standardized using StandardScaler, while categorical variables will be converted into numerical form using OneHotEncoder. This ensures that each type of feature receives the appropriate preprocessing

In [ ]:
numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

## Create preprocessing pipeline
I created a ColumnTransformer to preprocess different types of variables separately. Numerical features are standardized using StandardScaler, while categorical features are converted into numerical values using OneHotEncoder. This prepares all input variables in a suitable format for machine-learning algorithms.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        
        (
            "numerical",
            StandardScaler(),
            numeric_features
        ),
        
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

## Fit and transform training data
I applied the preprocessing pipeline to the training data using fit_transform(). The preprocessor learned the scaling parameters and categorical categories from the training data and then transformed the features into numerical form. The resulting training dataset contains 960 records and 50 processed features.

In [ ]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

print(
    "Processed training shape:",
    X_train_processed.shape
)

## Transform test data
The testing dataset was successfully transformed using the preprocessing parameters learned from the training data. Numerical features were scaled and categorical features were encoded consistently with the training dataset. The processed testing data contains 240 observations and 50 features, matching the feature structure of the processed training data

In [ ]:
X_test_processed = preprocessor.transform(
    X_test
)

print(
    "Processed testing shape:",
    X_test_processed.shape
)

## Convert processed data to DataFrame
After encoding and scaling the data, I retrieved the names of all transformed features and converted the processed training and testing arrays into Pandas DataFrames. This makes the transformed features easier to understand and verify before moving to machine-learning model training.

In [ ]:
feature_names = (
    preprocessor
    .get_feature_names_out()
)

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("Processed training data:")
display(X_train_processed_df.head())

## Final check
Finally, I performed a complete validation of my feature engineering and preprocessing process. I confirmed the dataset dimensions, training and testing split, processed feature dimensions, missing values, and target variable. The checks confirm that the data is clean, consistently transformed, and ready for machine-learning model training.

In [ ]:
print("\nOriginal dataset:")
print(df.shape)

print("\nTraining data:")
print(X_train.shape)

print("\nTesting data:")
print(X_test.shape)

print("\nProcessed training data:")
print(X_train_processed.shape)

print("\nProcessed testing data:")
print(X_test_processed.shape)

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nTarget variable:")
print(target)

print("\nFeature engineering and preprocessing completed!")

## Save the processed datasets
All processed features and target datasets were successfully saved as separate CSV files for the machine-learning stage.

In [ ]:
X_train_processed_df.to_csv(
    "X_train_processed.csv",
    index=False
)

X_test_processed_df.to_csv(
    "X_test_processed.csv",
    index=False
)

y_train.to_csv(
    "y_train.csv",
    index=False
)

y_test.to_csv(
    "y_test.csv",
    index=False
)

print("All processed files saved successfully!")

## Saving the Feature-Engineered Dataset
The complete feature-engineered dataset was saved as a CSV file for reuse in subsequent project stages

In [ ]:
df.to_csv("ev_vs_petrol_feature_engineered.csv", index=False)

## Feature Engineering & Preprocessing Conclusion

The feature engineering and preprocessing stage was successfully completed for the EV and Petrol vehicle dataset. The dataset was first examined for missing and infinite values, and the data quality checks confirmed that the dataset was suitable for further processing.

Several meaningful features were created to improve the analysis, including total vehicle sales, EV adoption measures, fuel-sales ratios, charging infrastructure indicators, EV range categories, fuel-price categories, subsidy categories, and charging-infrastructure categories.

To ensure reliable machine-learning results, features that could cause **target leakage** were identified and removed from the ML input data. The target variable was defined as **`ev_sales`**, while the remaining independent variables were used as model inputs.

The data was divided into **80% training and 20% testing sets**. Numerical features were standardized using **StandardScaler**, while categorical features were converted into numerical representations using **OneHotEncoder**. The preprocessing pipeline was fitted only on the training data and then applied to the testing data, helping prevent data leakage.

The processed training and testing datasets were successfully converted into clearly labelled DataFrames and saved as separate CSV files. The complete feature-engineered dataset was also saved as **`ev_vs_petrol_feature_engineered.csv`**.

Overall, the dataset is now clean, feature-engineered, properly preprocessed, and ready for the next stage: **machine-learning model training, comparison, tuning, prediction, and evaluation**.
